# 프로세스

문제 출처: [프로그래머스 — 프로세스](https://school.programmers.co.kr/learn/courses/30/lessons/42587)

[전체 목차](../../README.md) · [코테 목차](../README.md) · [스택·큐 목차](README.md)

정리일: 2026-10-06

## 문제

실행 대기 큐의 프로세스를 다음 규칙에 따라 처리합니다.

1. 맨 앞 프로세스를 꺼냅니다.
2. 남은 큐에 더 높은 우선순위가 있으면 꺼낸 프로세스를 맨 뒤에 넣습니다.
3. 더 높은 우선순위가 없으면 실행하고 종료합니다. 실행한 프로세스는 다시 넣지 않습니다.

`priorities`는 처음 대기 순서대로 나열한 우선순위입니다. `location`은 추적할 프로세스의 **처음 인덱스(0부터)**입니다. 해당 프로세스가 **몇 번째로 실행되는지(1부터)** 정수로 반환하세요. 뒤로 보내는 동작은 실행 횟수에 포함하지 않습니다.

### 제한사항

- `priorities`의 길이는 1 이상 100 이하입니다.
- 우선순위는 1 이상 9 이하인 정수이며, 클수록 먼저 실행합니다.
- `location`은 0 이상 `len(priorities) - 1` 이하입니다.

### 입출력 예

| priorities | location | 반환값 |
| --- | --- | --- |
| `[2, 1, 3, 2]` | `2` | `1` |
| `[1, 1, 9, 1, 1, 1]` | `0` | `5` |

첫 예제에서 처음 프로세스를 A, B, C, D라고 하면 실행 순서는 C → D → A → B입니다. `location=2`인 C는 첫 번째로 실행됩니다.

두 번째 예제의 실행 순서는 C → D → E → F → A → B입니다. `location=0`인 A는 다섯 번째로 실행됩니다.


## 다시 풀기

직접 풀이를 작성한 뒤 예제 셀을 실행하세요. 목표 프로세스가 맨 앞에 온 것과 실제로 실행되는 것을 구분해 보세요.


In [ ]:
from collections import deque


def solution(priorities, location):
    pass


In [ ]:
print(solution([2, 1, 3, 2], 2))  # 예상 출력: 1
print(solution([1, 1, 9, 1, 1, 1], 0))  # 예상 출력: 5


## 해설

### 아이디어 진단

“현재 최댓값이면 실행하고, 아니면 맨 앞 원소를 뒤로 보낸다”는 접근은 맞았습니다. `que.append(que.popleft())`도 올바른 큐 회전 코드입니다. 구현 과정에서는 튜플과 정수의 비교, 실행 시점, 인덱싱에서 막혔습니다. 마지막에는 `max(i[1] for i in que)`를 “튜플을 하나씩 받아 인덱스 1번 값을 본다”고 직접 설명했습니다. 최종 독립 풀이와 플랫폼 제출 통과는 확인하지 않았습니다.

### 핵심 흐름

1. `deque(enumerate(priorities))`로 `(원래 인덱스, 우선순위)`를 함께 보관합니다.
2. 현재 큐의 우선순위 최댓값을 구합니다.
3. 맨 앞 우선순위가 최댓값이면 실행 횟수를 1 늘립니다. 그 원래 인덱스가 `location`이면 반환하고, 아니면 맨 앞 원소를 제거합니다.
4. 최댓값이 아니면 맨 앞 원소를 뒤로 보냅니다.

원래 인덱스는 이동해도 바뀌지 않는 이름표입니다. 우선순위가 같은 프로세스도 구분할 수 있습니다. 단순 내림차순 정렬은 이동 후 같은 우선순위끼리의 대기 순서를 반영하지 못합니다.

### 실수 1: 큐의 원소와 튜플 내부 값 구분

```python
que = deque([(0, 2), (1, 1), (2, 3), (3, 2)])
```

| 표현 | 값 | 타입 / 의미 |
| --- | --- | --- |
| `que[0]` | `(0, 2)` | tuple / 맨 앞 프로세스 |
| `que[1]` | `(1, 1)` | tuple / 두 번째 프로세스 |
| `que[0][0]` | `0` | int / 맨 앞 프로세스의 원래 번호 |
| `que[0][1]` | `2` | int / 맨 앞 프로세스의 우선순위 |
| `max(que)` | `(3, 2)` | tuple / 첫 항목인 원래 번호부터 비교한 결과 |
| `max(que[1])` | `1` | int / 두 번째 튜플 `(1, 1)` 안에서 구한 최댓값 |

`que[0[0]]`은 숫자 `0`에 인덱싱을 시도하므로 오류입니다. `que[0][0]`처럼 바깥 원소를 고르고 그 안의 값을 고릅니다.

`que[0][1] == max(que)`는 정수와 튜플을 비교하므로 거짓입니다. 이 때문에 실행 횟수가 늘지 않은 채 회전하다가, 맨 앞 원래 번호만 검사하는 반환문에서 0을 반환했습니다.

### 우선순위만 뽑아 max에 전달하기

```python
highest = max(i[1] for i in que)
```

`i`는 인덱스 숫자가 아니라 튜플 하나입니다. `item`으로 이름을 바꿔도 같습니다. 괄호 안 제너레이터 표현식이 우선순위를 하나씩 `max`에 제공합니다. 큐를 읽기만 하며 제거하지 않습니다.

| 순회 중 i | i[1] |
| --- | --- |
| `(0, 2)` | `2` |
| `(1, 1)` | `1` |
| `(2, 3)` | `3` |
| `(3, 2)` | `2` |

따라서 `highest`는 정수 `3`입니다. 일반 반복문으로 풀면 다음과 같습니다.

```python
values = []
for i in que:
    values.append(i[1])
highest = max(values)
print(highest)  # 예상 출력: 3
```

`max(que, key=lambda i: i[1])`는 우선순위로 비교하되 **튜플 전체**를 반환합니다. 이 문제는 숫자만 필요하므로 `max(i[1] for i in que)`를 사용합니다. `any()`를 추가로 쓰지 않아도 됩니다.

### 실수 2: 맨 앞에 도착한 시점과 실행 시점 구분

`que[0][0] == location`만 먼저 확인하면 안 됩니다. 목표 프로세스가 맨 앞이어도 더 높은 우선순위가 남아 있으면 뒤로 보내야 합니다.

**실행 가능 여부 확인 → answer += 1 → 목표인지 확인** 순서입니다. 모범 코드에서는 목표라면 즉시 함수가 끝나므로 반환 직전에 별도로 `popleft()`를 하지 않아도 됩니다.

### 실수 3: 메서드 호출과 조건부 회전

```python
que.append(que.popleft())
```

안쪽 `popleft()`가 맨 앞 튜플을 제거해서 반환하고, 바깥 `append()`가 그 튜플을 뒤에 넣습니다. 아래와 같습니다.

```python
process = que.popleft()
que.append(process)
```

`que.popleft`만 쓰면 메서드를 가리킬 뿐 실행되지 않습니다. 괄호 `()`가 필요합니다. 뒤로 보내는 줄은 반드시 `else`에 둡니다. 실행한 직후 다음 프로세스까지 검사 없이 이동시키면 안 됩니다.

`while que:`에 진입한 직후는 빈 큐가 아니므로 `que[0]`과 `max`를 사용할 수 있습니다. 반면 `que and que[1]`은 안전하지 않습니다. 원소가 하나 있어도 `que`는 참이지만 두 번째 원소는 없기 때문입니다.

### 값 추적: priorities = [2, 1, 3, 2], location = 0

A는 원래 번호 0인 프로세스입니다. 뒤로 보내도 이름표는 유지됩니다.

| 처리 전 큐: 원래 번호와 우선순위 | 최댓값 | 동작 | answer |
| --- | --- | --- | --- |
| `[(0,2), (1,1), (2,3), (3,2)]` | 3 | A를 뒤로 | 0 |
| `[(1,1), (2,3), (3,2), (0,2)]` | 3 | B를 뒤로 | 0 |
| `[(2,3), (3,2), (0,2), (1,1)]` | 3 | C 실행 및 제거 | 1 |
| `[(3,2), (0,2), (1,1)]` | 2 | D 실행 및 제거 | 2 |
| `[(0,2), (1,1)]` | 2 | A 실행, 원래 번호가 location과 같으므로 반환 | 3 |

### 필요한 Python 도구

이건 코테에서 자주 쓰니 챙겨라: **enumerate로 원래 번호 붙이기**, **deque로 앞에서 꺼내 뒤에 넣기**, **max에 필요한 값만 전달하기**.

- `enumerate(priorities)`는 `(인덱스, 값)` 튜플을 제공하는 이터레이터입니다. 기본 인덱스는 0이며 입력 리스트를 바꾸지 않습니다. 빈 입력이면 제공할 튜플이 없습니다.
- `deque(...)`는 순회 가능한 입력으로 새 큐를 만듭니다. 입력 리스트의 순서는 바꾸지 않습니다.
- `popleft()`는 맨 앞 원소를 제거하고 반환합니다. 여기서는 튜플을 반환합니다. 빈 큐에서는 `IndexError`가 납니다.
- `append(x)`는 맨 뒤에 원소를 추가하며 `None`을 반환합니다. 빈 큐에도 추가할 수 있습니다.
- `max(...)`는 최댓값을 반환합니다. 여기서는 우선순위 정수입니다. 원본 큐는 바꾸지 않으며, 기본값 없이 빈 입력에 호출하면 `ValueError`가 납니다.

자세한 사용법: [스택·큐 기본 문법](스택큐-사용법.ipynb) · [enumerate](../../01-파이썬-도구/04-반복과-변환/enumerate.ipynb) · [min / max](../../01-파이썬-도구/01-숫자/len-sum-min-max.md)

### 경계 조건과 복잡도

- 프로세스가 하나면 첫 실행에서 1을 반환합니다.
- 우선순위가 모두 같으면 처음 순서대로 실행하므로 `location + 1`을 반환합니다.
- 큐가 비었을 때는 `while` 본문에 들어가지 않으므로 빈 입력에 `max`를 호출하지 않습니다. 문제에서는 처음 길이가 1 이상입니다.
- `n`은 프로세스 수입니다. 매번 `max`는 현재 큐 길이에 비례하고, 양끝 제거·추가는 O(1)입니다. 우선순위가 1~9로 제한되므로 각 우선순위 단계에서 한 바퀴 이내의 회전 후 해당 최댓값들이 실행됩니다. 전체 반복 횟수는 O(9n), 따라서 시간은 O(n²), 큐 공간은 O(n)입니다.

### 가져갈 것

1. `que[0][0]`은 원래 번호, `que[0][1]`은 우선순위다.
2. `max(i[1] for i in que)`의 `i`는 튜플이며, 각 우선순위를 읽어 최댓값을 구한다.
3. 실행할 수 있을 때만 횟수를 늘리고 목표인지 확인한다. 실행 못 하면 앞에서 꺼내 뒤에 넣는다.


## 모범 코드

대화에서 선택한 `max` 방식과 꺼내기 전 목표 번호 확인을 유지했습니다. `solution_model`은 직접 풀 함수 `solution`을 덮어쓰지 않습니다.


In [ ]:
from collections import deque


def solution_model(priorities, location):
    answer = 0
    que = deque(enumerate(priorities))

    while que:
        highest = max(i[1] for i in que)

        if que[0][1] == highest:
            answer += 1

            if que[0][0] == location:
                return answer

            que.popleft()
        else:
            que.append(que.popleft())


print(solution_model([2, 1, 3, 2], 2))  # 예상 출력: 1
print(solution_model([1, 1, 9, 1, 1, 1], 0))  # 예상 출력: 5
print(solution_model([2, 1, 3, 2], 0))  # 예상 출력: 3
print(solution_model([7], 0))  # 예상 출력: 1
print(solution_model([2, 2, 2], 1))  # 예상 출력: 2
